In [1]:
import re, time
import numpy as np
import pandas as pd
import torch
from transformers import pipeline

torch.set_num_threads(2)   # same as the latency benchmark
sentiment = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    top_k=None, truncation=True, max_length=512, device="cpu",
)

def light_clean(t: str) -> str:
    """Remove CFPB redactions but keep capitals and punctuation,
    which carry emotional intensity (e.g. 'NEVER', '!!!')."""
    return re.sub(r"\s+", " ", re.sub(r"(?i)x{2,}", " ", t)).strip()

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [2]:
val = pd.read_csv("../data/val.csv").sample(500, random_state=42).reset_index(drop=True)

results = sentiment([light_clean(t) for t in val["text"]], batch_size=16)
scores = pd.DataFrame([{d["label"]: d["score"] for d in r} for r in results])
val = pd.concat([val, scores], axis=1)
val["top"] = scores.idxmax(axis=1)

print(val["top"].value_counts(normalize=True))
print(val["negative"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

top
negative    0.722
neutral     0.274
positive    0.004
Name: proportion, dtype: float64
count    500.000000
mean       0.612983
std        0.261875
min        0.012692
10%        0.172585
25%        0.423136
50%        0.712419
75%        0.820128
90%        0.878583
max        0.938036
Name: negative, dtype: float64


In [3]:
pd.set_option("display.max_colwidth", 300)
print("MOST negative:")
print(val.nlargest(3, "negative")[["label", "negative", "text"]])
print("\nLEAST negative:")
print(val.nsmallest(3, "negative")[["label", "negative", "text"]])

MOST negative:
             label  negative  \
18   report_misuse  0.938036   
97  consumer_loans  0.937692   
40   report_misuse  0.931565   

                                                                                                                                                                                                                                                                                                           text  
18                                                                                                                                            equifax was hacked in 2017 and waited for 5 weeks to let those affected know they were at risk. i am now susceptible to credit fraud because of their negligence.  
97  they are overcharging me. i ca n't get ahead. they are not helpful in any way. they will not work with me so that i can repay what i owe. due to my financial circumstances and being a single mother it is already a struggle to repay and they

In [4]:
texts = [light_clean(t) for t in val["text"][:200]]
for t in texts[:10]:
    sentiment(t)                     # warm-up

times = []
for t in texts:
    start = time.perf_counter()
    sentiment(t)
    times.append((time.perf_counter() - start) * 1000)

t = np.array(times)
print(f"sentiment  p50={np.percentile(t, 50):.1f} ms  p95={np.percentile(t, 95):.1f} ms  max={t.max():.1f} ms")

sentiment  p50=69.8 ms  p95=182.2 ms  max=191.0 ms


In [5]:
for max_len in (128, 256):
    pipe = pipeline("sentiment-analysis",
                    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                    top_k=None, truncation=True, max_length=max_len, device="cpu")
    for t in texts[:10]:
        pipe(t)
    times = []
    for t in texts:
        start = time.perf_counter()
        pipe(t)
        times.append((time.perf_counter() - start) * 1000)
    a = np.array(times)
    print(f"max_length={max_len}: p50={np.percentile(a,50):.1f} ms  "
          f"p95={np.percentile(a,95):.1f} ms  max={a.max():.1f} ms")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


max_length=128: p50=51.0 ms  p95=55.7 ms  max=85.0 ms


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


max_length=256: p50=71.2 ms  p95=105.0 ms  max=143.1 ms


In [6]:
pipe128 = pipeline("sentiment-analysis",
                   model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                   top_k=None, truncation=True, max_length=128, device="cpu")

res128 = pipe128([light_clean(t) for t in val["text"]], batch_size=16)
val["negative_128"] = [ {d["label"]: d["score"] for d in r}["negative"] for r in res128 ]

print("Correlation with 512-token score:", val["negative"].corr(val["negative_128"]).round(3))
print("Mean absolute difference:", (val["negative"] - val["negative_128"]).abs().mean().round(3))
print("Label agreement:", (val["negative_128"].round() == val["negative"].round()).mean().round(3))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Correlation with 512-token score: 0.987
Mean absolute difference: 0.025
Label agreement: 0.972
